# **BUILDING AND TRAINING A MINIGPT-V2 ON CONCEPTS BEYOND 2017 (RMSnorm, SwiGLU, RoPE, GQA, LR-Scheduling) and SIMPLE BPE-TOKENIZER**

## **Step 1: Dataset — TinyStories**

In [ ]:
from datasets import load_dataset

# Streaming avoids downloading the full 2.1M-story dataset
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# Subsample — 15,000 stories is plenty of repetitive structure
# without blowing past Colab free-tier session limits
num_stories = 15000
stories = []
for i, example in enumerate(ds):
    if i >= num_stories:
        break
    stories.append(example["text"])

text = "\n".join(stories)
print(f"Total characters: {len(text):,}")
print(f"First 300 chars:\n{text[:300]}")

## **Step 2: BPE Tokenizer**

In [ ]:
def get_stats(ids):
    """
    Count how often every adjacent pair occurs.
    ids: list of ints (current token sequence)
    Returns: dict {(id1, id2): count}
    """
    counts = {}
    for pair in zip(ids, ids[1:]):          # (ids[0],ids[1]), (ids[1],ids[2]), ...
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    """
    Replace every occurrence of `pair` in ids with a single new_id.
    ids: list of ints
    pair: (id1, id2) tuple to merge
    new_id: the new token id to substitute
    """
    new_ids = []
    i = 0
    while i < len(ids):
        # Check if this pair matches at position i
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2                           # skip both merged tokens
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

In [ ]:
# import re

# def split_on_whitespace(text):
#     return re.findall(r'\s+|\S+', text)

# def get_stats_multi(chunks, counts=None):
#     counts = {} if counts is None else counts
#     for chunk_ids in chunks:
#         for pair in zip(chunk_ids, chunk_ids[1:]):
#             counts[pair] = counts.get(pair, 0) + 1
#     return counts

# def merge_multi(chunks, pair, new_id):
#     return [merge(chunk_ids, pair, new_id) for chunk_ids in chunks]

In [ ]:
def train_bpe(text, vocab_size):
    """
    text: full training corpus (string)
    vocab_size: target vocab size (must be > 256)
    Returns: merges dict {(id1,id2): new_id}, in the order they were learned
    """
    num_merges = vocab_size - 256
    ids = list(text.encode("utf-8"))   # start: raw bytes, 0-255

    merges = {}   # (id1, id2) -> new_id
    for i in range(num_merges):
        stats = get_stats(ids)
        if not stats:
            break                       # nothing left to merge
        pair = max(stats, key=stats.get)   # most frequent pair
        new_id = 256 + i                    # next available id
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id

        if i % 100 == 0:
            print(f"merge {i}: {pair} -> {new_id} (had {stats[pair]} occurrences)")

    return merges, ids

In [ ]:
def bpe_encode(text, merges):
    """
    Encodes text using whitespace-respecting merges.
    Splits into chunks first, encodes each chunk independently,
    then concatenates the results.
    """
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def bpe_decode(ids, merges):
    """
    Reverse the merges to get back raw bytes, then decode to text.
    """
    # Build id -> bytes mapping, starting from the 256 raw bytes
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in merges.items():
        vocab[idx] = vocab[p0] + vocab[p1]   # concatenate the two pieces

    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8", errors="replace")

In [ ]:
merges, chunks = train_bpe(
    text,
    vocab_size=2000,     # real cap — identical to original run, isolates the variable
)
vocab_size = 256 + len(merges)
print(f"Vocab size: {vocab_size}")

encode = lambda s: bpe_encode(s, merges)
decode = lambda l: bpe_decode(l, merges)

# Encode full dataset
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print(f"Total merges learned: {len(merges)}")
print(f"Final vocab size: {256 + len(merges)}\n")

# Look at the tail, whatever size it ended up being
tail_size = 100
for (p0, p1), idx in list(merges.items())[-tail_size:]:
    piece = vocab[idx].decode("utf-8", errors="replace")
    print(f"{idx}: {repr(piece)}  (merged from {p0},{p1})")

# The actual correctness check: flag anything that looks like a phrase-crossing token
print("\n--- Checking for any boundary violations ---")
violations = 0
for idx, piece_bytes in vocab.items():
    if idx < 256:
        continue
    piece = piece_bytes.decode("utf-8", errors="replace")
    stripped = piece.strip()
    if " " in stripped:   # a space survives even after stripping leading/trailing = it's in the middle
        print(f"VIOLATION — token {idx}: {repr(piece)}")
        violations += 1

print(f"\n{violations} boundary violations found out of {len(merges)} merges")

## **IMPORTS**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import requests

## **Step 3: Train/Val Split and Hyperparameters**

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
# 90% train, 10% validation
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens  : {len(val_data):,}")

block_size = 64
batch_size = 32
d_model    = 256    # up from 192 — 4/3× wider
n_heads    = 4      # unchanged: 256/4 = 64 per head, clean
n_kv_heads = 2      # unchanged — still valid (4 heads / 2 groups = 2:1)
d_ff       = 683    # SwiGLU 2/3 scaling: 256 × 4 × (2/3) ≈ 683
n_layers   = 8      # up from 6

dropout    = 0.1
lr         = 3e-4

max_iters  = 10000   # doubled — training is cheap, and more steps directly
                     # targets the token-boundary problem, not just capacity
eval_every = 200     # unchanged, just means more total print lines

# LR schedule additions
warmup_steps = int(0.1 * max_iters)   # recompute — stays 10% = 1000 now
min_lr       = lr * 0.1

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

## **Step 4: The Batch Loader**

In [ ]:
# ============================================================
# Step 4: Batch Loader (unchanged)
# ============================================================
def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i   : i+block_size  ] for i in ix])
    y = torch.stack([data_split[i+1 : i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## **Step 5: Rotary Positional Encoding (RoPE)**

In [ ]:
def precompute_rope_freqs(d_k, max_seq_len, base=10000, device='cpu'):
    """
    Precomputes the cosine and sine tables used for RoPE rotation.

    d_k        : dimension per head (e.g. 32 for d_model=128, n_heads=4)
    max_seq_len: maximum sequence length we'll ever see (use block_size here)
    base       : controls rotation speeds — 10000 is the standard default
                 (same value used in the original sinusoidal PE denominator)

    Returns:
        cos_table : (max_seq_len, d_k/2)  — one cos value per position per pair
        sin_table : (max_seq_len, d_k/2)  — one sin value per position per pair
    """

    # Step 1: compute one base angle (theta) per dimension PAIR
    # d_k=32 → 16 pairs → 16 theta values
    # Formula: theta_i = 1 / (base ^ (2i / d_k))  for i = 0, 1, ..., d_k/2 - 1
    # i=0  → theta = 1/10000^0       = 1.0      (fastest rotation)
    # i=1  → theta = 1/10000^(2/32)  = 0.66     (slightly slower)
    # ...
    # i=15 → theta = 1/10000^(30/32) = 0.000126 (slowest rotation)
    pair_indices = torch.arange(0, d_k, 2, device=device).float()  # [0, 2, 4, ..., d_k-2]
    thetas       = 1.0 / (base ** (pair_indices / d_k))             # (d_k/2,)

    # Step 2: for every token position, multiply by its theta
    # positions: [0, 1, 2, ..., max_seq_len-1]
    # result[pos, i] = pos * theta_i  ← rotation angle for token at 'pos', pair 'i'
    positions = torch.arange(max_seq_len, device=device).float()    # (max_seq_len,)
    angles    = torch.outer(positions, thetas)                       # (max_seq_len, d_k/2)
    # torch.outer: multiplies every position by every theta
    # angles[3, 1] = 3 * theta_1  → rotation angle for position 3, pair 1

    # Step 3: precompute cos and sin of every angle
    # (so we don't recompute them from scratch on every forward pass)
    cos_table = angles.cos()   # (max_seq_len, d_k/2)
    sin_table = angles.sin()   # (max_seq_len, d_k/2)

    return cos_table, sin_table

In [ ]:
def apply_rope(x, cos_table, sin_table, start_pos=0):
    """
    Rotates Q or K vectors using precomputed RoPE tables.

    x          : (B, num_heads, T, d_k) — Q or K after splitting into heads
    cos_table  : (max_seq_len, d_k/2)
    sin_table  : (max_seq_len, d_k/2)
    start_pos  : where in the sequence does this input start
                 (0 for training/prefill, grows during decode with KV cache)

    Returns:
        x_rotated : (B, num_heads, T, d_k) — same shape, rotated values
    """
    B, num_heads, T, d_k = x.shape

    # Step 1: slice out only the rows of cos/sin we need
    # (positions start_pos through start_pos + T - 1)
    cos = cos_table[start_pos : start_pos + T]   # (T, d_k/2)
    sin = sin_table[start_pos : start_pos + T]   # (T, d_k/2)

    # Step 2: split x into consecutive pairs along the last dimension
    # d_k=32 → x_even = dims [0,2,4,...,30], x_odd = dims [1,3,5,...,31]
    x_even = x[..., 0::2]   # (B, num_heads, T, d_k/2) — first of each pair
    x_odd  = x[..., 1::2]   # (B, num_heads, T, d_k/2) — second of each pair

    # Step 3: apply the 2D rotation formula to each pair
    # Standard 2D rotation of vector [a, b] by angle α:
    #   new_a = a*cos(α) - b*sin(α)
    #   new_b = a*sin(α) + b*cos(α)
    x_rotated_even = x_even * cos - x_odd * sin   # (B, num_heads, T, d_k/2)
    x_rotated_odd  = x_even * sin + x_odd * cos   # (B, num_heads, T, d_k/2)

    # Step 4: interleave even and odd back into one tensor
    # Stack along a new last dim → (B, num_heads, T, d_k/2, 2)
    # then flatten last two dims → (B, num_heads, T, d_k)
    x_rotated = torch.stack([x_rotated_even, x_rotated_odd], dim=-1)
    x_rotated = x_rotated.flatten(-2)             # (B, num_heads, T, d_k)

    return x_rotated

## **Step 6: GQA (Grouped Query Attention)**

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, n_kv_heads, dropout, max_seq_len):
        super().__init__()
        assert d_model % num_heads == 0,   "d_model must be divisible by num_heads"
        assert num_heads % n_kv_heads == 0, "num_heads must be divisible by n_kv_heads"

        self.d_model          = d_model
        self.num_heads        = num_heads
        self.n_kv_heads       = n_kv_heads
        self.heads_per_group  = num_heads // n_kv_heads  # Q heads sharing each KV pair
        self.d_k              = d_model // num_heads     # dim per head

        # W_Q projects to full num_heads dimension — every Q head is independent
        self.W_Q = nn.Linear(d_model, num_heads  * self.d_k, bias=False)
        # W_K and W_V project to n_kv_heads dimension — smaller than before
        # e.g. d_model=128, n_kv_heads=2, d_k=32 → projects to 64, not 128
        self.W_K = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

        # RoPE tables — precomputed once, fixed (not learned), moved to GPU with model
        cos_table, sin_table = precompute_rope_freqs(self.d_k, max_seq_len)
        self.register_buffer('cos_table', cos_table)  # (max_seq_len, d_k/2)
        self.register_buffer('sin_table', sin_table)  # (max_seq_len, d_k/2)

    def forward(self, x, kv_cache=None, start_pos=0):
        """
        x shape:
          Training / prefill : (B, T, d_model)
          Decode (1 new token): (B, 1, d_model)

        kv_cache: dict with 'K' and 'V' of shape (B, n_kv_heads, T_past, d_k)
                  or None on the first call.
        start_pos: position index where this input starts in the full sequence.

        Returns:
          output       : (B, T, d_model) — same shape as input
          new_kv_cache : updated cache dict with compact (n_kv_heads) K, V
        """
        B, T, C = x.shape

        # ── Project Q, K, V ─────────────────────────────────────────────────
        Q = self.W_Q(x)   # (B, T, num_heads * d_k)  = (B, T, 128)
        K = self.W_K(x)   # (B, T, n_kv_heads * d_k) = (B, T, 64)  ← smaller
        V = self.W_V(x)   # (B, T, n_kv_heads * d_k) = (B, T, 64)  ← smaller

        # ── Reshape into heads ───────────────────────────────────────────────
        # Q: split into num_heads (4) heads of size d_k (32)
        Q = Q.view(B, T, self.num_heads,  self.d_k).transpose(1, 2)  # (B, 4, T, 32)
        # K, V: split into n_kv_heads (2) groups of size d_k (32)
        K = K.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)  # (B, 2, T, 32)
        V = V.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)  # (B, 2, T, 32)

        # ── Apply RoPE to Q and K ────────────────────────────────────────────
        # Q gets 4-head rotation, K gets 2-head rotation.
        # Both use the same start_pos — same sequence positions, just different
        # numbers of heads. apply_rope reads shape from the tensor automatically.
        Q = apply_rope(Q, self.cos_table, self.sin_table, start_pos)  # (B, 4, T, 32)
        K = apply_rope(K, self.cos_table, self.sin_table, start_pos)  # (B, 2, T, 32)

        # ── KV Cache — store COMPACT K, V (n_kv_heads, not num_heads) ────────
        # The cache stores only 2 heads worth of K, V, not 4.
        # This is the memory saving — expansion happens at attention time below.
        if kv_cache is not None:
            K = torch.cat([kv_cache['K'], K], dim=2)   # (B, 2, T_past+T, 32)
            V = torch.cat([kv_cache['V'], V], dim=2)   # (B, 2, T_past+T, 32)

        new_kv_cache = {'K': K, 'V': V}   # shape: (B, n_kv_heads=2, full_len, d_k)
        full_len = K.shape[2]

        # ── Expand K, V to match num_heads before dot product ────────────────
        # repeat_interleave(n, dim=1) repeats each "slice" along dim 1, n times.
        # (B, 2, full_len, 32) → (B, 4, full_len, 32)
        # K_group0 becomes [K_group0, K_group0], K_group1 becomes [K_group1, K_group1]
        # so Q_head0 and Q_head1 both see K_group0, Q_head2 and Q_head3 see K_group1.
        K = K.repeat_interleave(self.heads_per_group, dim=1)  # (B, 4, full_len, 32)
        V = V.repeat_interleave(self.heads_per_group, dim=1)  # (B, 4, full_len, 32)

        # ── Everything below is identical to MHA ─────────────────────────────
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, 4, T, full_len)

        if T > 1:
            mask = torch.triu(
                torch.ones(T, full_len, device=x.device),
                diagonal=full_len - T + 1
            ).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = attn_weights @ V                            # (B, 4, T, 32)

        # ── Merge heads back ─────────────────────────────────────────────────
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, 128)
        return self.W_O(out), new_kv_cache

## **Step 7: Feed Forward Network (FFN) - SwiGLU**

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.W1      = nn.Linear(d_model, d_ff, bias=False)  # gate path
        self.W2      = nn.Linear(d_model, d_ff, bias=False)  # value path
        self.W3      = nn.Linear(d_ff, d_model, bias=False)  # contract
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Gate path: Swish activation (F.silu is PyTorch's name for Swish = x * sigmoid(x))
        gate  = F.silu(self.W1(x))   # (B, T, d_ff) — decides what to amplify
        # Value path: no activation — carries the raw projected content
        value = self.W2(x)            # (B, T, d_ff)
        # Gated multiplication: gate controls how much of value passes through
        x = gate * value              # (B, T, d_ff) — elementwise
        # Contract back to d_model
        x = self.W3(x)                # (B, T, d_model)
        return self.dropout(x)

## **Step 8: RMSNorm**

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # learned scale

    def forward(self, x):
        # x shape: (..., d_model) — works for (B, T, d_model) during
        # training/prefill AND (B, 1, d_model) during single-token decode.
        rms    = x.pow(2).mean(dim=-1, keepdim=True).sqrt()   # RMS over last dim
        x_norm = x / (rms + self.eps)                         # re-scale only
        return self.gamma * x_norm                            # learned scale

## **Step 9: The Transformer Block**

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, n_kv_heads, d_ff, dropout, max_seq_len):
        super().__init__()
        self.attention    = GroupedQueryAttention(d_model, num_heads, n_kv_heads, dropout, max_seq_len)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1        = RMSNorm(d_model)
        self.norm2        = RMSNorm(d_model)

    def forward(self, x, kv_cache=None, start_pos=0):
        # Pre-LN: normalize BEFORE feeding into sublayer
        attn_out, new_kv_cache = self.attention(self.norm1(x), kv_cache=kv_cache, start_pos=start_pos)
        x = x + attn_out                  # residual connection

        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out                    # residual connection

        return x, new_kv_cache

## **Step 10: The Full MiniGPT Model**

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, n_kv_heads, d_ff, num_layers, dropout, max_seq_len):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, num_heads, n_kv_heads, d_ff, dropout, max_seq_len)
            for _ in range(num_layers)
        ])
        self.final_norm  = RMSNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, kv_caches=None, start_pos=0):
        """
        token_ids: (B, T)  during training
                   (B, T)  during prefill  (T = prompt length)
                   (B, 1)  during decode   (one new token per step)

        kv_caches: list of num_layers cache dicts, or None
        start_pos: where in the full sequence does this input begin
                   (0 for training/prefill, grows during decode)

        Returns:
            logits     : (B, T, vocab_size)
            kv_caches  : updated list of per-block caches
        """
        B, T = token_ids.shape

        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        # Step 1: Embed tokens → (B, T, d_model)
        x = self.embedding(token_ids)
        x = self.emb_dropout(x)

        # NOTE: No positional encoding addition here anymore.
        # RoPE handles position inside each attention layer instead.
        # pe               = positional_encoding(start_pos + T, self.embedding.embedding_dim, token_ids.device)
        # pe_slice         = pe[start_pos : start_pos + T]    # (T, d_model)
        # x                = x + pe_slice                     # broadcasts over batch dim

        # Step 2: Pass through transformer blocks
        new_kv_caches = []
        for block, block_cache in zip(self.blocks, kv_caches):
            x, new_block_cache = block(x, kv_cache=block_cache, start_pos=start_pos)
            new_kv_caches.append(new_block_cache)

        # Step 3: Final norm + output head
        x      = self.final_norm(x)
        logits = self.output_head(x)      # (B, T, vocab_size)

        return logits, new_kv_caches

## **Step 11: Instantiate Model**

In [ ]:
model = MiniGPT(
    vocab_size  = vocab_size,   # now ~2000, not 65
    d_model     = d_model,
    num_heads   = n_heads,
    n_kv_heads  = n_kv_heads,
    d_ff        = d_ff,
    num_layers  = n_layers,
    dropout     = dropout,
    max_seq_len = block_size,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## **Step 12: LR Scheduling & Helpers**

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn   = nn.CrossEntropyLoss()

def get_lr(step):
    if step < warmup_steps:
        return lr * (step / warmup_steps)
    progress = (step - warmup_steps) / (max_iters - warmup_steps)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * progress))

## **Step 13: Training Loop**

In [ ]:
@torch.no_grad()
def estimate_val_loss(eval_batches=20):
    model.eval()
    total_loss = 0.0
    for _ in range(eval_batches):
        xb, yb    = get_batch('val')
        logits, _ = model(xb)
        loss      = loss_fn(logits.view(-1, vocab_size), yb.view(-1))
        total_loss += loss.item()
    model.train()
    return total_loss / eval_batches

print("Starting training...\n")

for step in range(max_iters):
    xb, yb = get_batch('train')

    current_lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    logits, _ = model(xb)
    loss = loss_fn(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_every == 0 or step == max_iters - 1:
        val_loss = estimate_val_loss()
        print(f"Step {step:4d} | LR: {current_lr:.6f} | Train loss: {loss.item():.4f} | Val loss: {val_loss:.4f}")

print("\nTraining complete.")

## **Step 12: Save the Model Weights and BPE Merges**

In [ ]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'vocab_size'        : vocab_size,
    'd_model'           : d_model,
    'n_heads'           : n_heads,
    'n_kv_heads'        : n_kv_heads,
    'd_ff'              : d_ff,
    'n_layers'          : n_layers,
    'dropout'           : dropout,
    'block_size'        : block_size,
    'merges'            : merges,
}, 'minigpt_v2.pt')

print("Model saved to minigpt_v2.pt")

## **Step 13: The Generation Flow**

In [ ]:
# ==============================================================================
# TOP-P APPROACH
# ==============================================================================

def _sample(logits_1d, temperature, top_p=0.9, generated=None, rep_penalty=1.2):
    """
    Temperature + repetition penalty + nucleus (top-p) sampling.
    """
    # Step 1: Repetition penalty — suppress recently seen characters
    if generated and rep_penalty > 1.0:
        for token_id in set(generated[-15:]):
            logits_1d[token_id] /= rep_penalty

    # Temperature = 0 → pure greedy
    if temperature == 0:
        return logits_1d.argmax().item()

    # Step 2: Temperature scaling
    logits_1d = logits_1d / temperature

    # Step 3: Convert to probabilities
    probs = torch.softmax(logits_1d, dim=-1)

    # Step 4: Top-p nucleus sampling
    # Sort probabilities descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)

    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Find cutoff: remove tokens once cumulative prob exceeds p
    # shift by 1 so we always keep at least the top token
    sorted_indices_to_remove = cumulative_probs - sorted_probs > top_p

    # Zero out the removed tokens
    sorted_probs[sorted_indices_to_remove] = 0.0

    # Renormalise so remaining probs sum to 1
    sorted_probs = sorted_probs / sorted_probs.sum()

    # Sample from the nucleus
    sampled_idx = torch.multinomial(sorted_probs, num_samples=1)

    # Map back to original token index
    return sorted_indices[sampled_idx].item()

In [ ]:
def generate(model, prompt, max_new_tokens=200, temperature=0.6, top_p=0.9, rep_penalty=1.2):
    model.eval()
    token_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    generated = token_ids[0].tolist()

    with torch.no_grad():

        # Prefill
        logits, kv_caches = model(token_ids, kv_caches=None, start_pos=0)
        next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                          generated=generated, rep_penalty=rep_penalty)
        generated.append(next_id)

        # Decode
        for _ in range(max_new_tokens - 1):
            current_pos  = len(generated) - 1
            input_tensor = torch.tensor([[next_id]], dtype=torch.long).to(device)
            logits, kv_caches = model(input_tensor, kv_caches=kv_caches,
                                      start_pos=current_pos)
            next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                              generated=generated, rep_penalty=rep_penalty)
            generated.append(next_id)

    return decode(generated)

## **Step 14: Generation**

In [ ]:
print("\n=== Sample generation ===\n")
output = generate(
    model,
    prompt         = "Once upon a time",
    max_new_tokens = 200,
    temperature    = 0.6,
    top_p          = 0.9,
    rep_penalty    = 1.2
)
print(output)

In [ ]:
# print(list(merges.items())[1600:1744])

In [ ]:
# # Decode every merge in the tail range and print what it actually spells
# vocab = {idx: bytes([idx]) for idx in range(256)}
# for (p0, p1), idx in merges.items():
#     vocab[idx] = vocab[p0] + vocab[p1]

# for (p0, p1), idx in list(merges.items())[1600:1744]:
#     piece = vocab[idx].decode("utf-8", errors="replace")
#     print(f"{idx}: {repr(piece)}  (merged from {p0},{p1})")